# 03 — Graph Analytics with Neo4j GDS

## Movie Collaboration Network Analysis

This notebook runs graph algorithms on the movie collaboration network using the Neo4j Graph Data Science (GDS) library.

### What we do here:
1. Connect to Neo4j and verify the graph is loaded
2. Project the graph into GDS memory
3. Run **PageRank** — to score structural importance of Directors and Actors
4. Run **Louvain Community Detection** — to find collaboration clusters
5. Inspect results and interpret them in business terms
6. Export features for the ML enrichment step in `04_ml.ipynb`

### GDS Workflow (3 steps, every algorithm):
```
Project → Run Algorithm → Drop Projection
```
Results are written back to the live graph as node properties.

## Step 0 — Imports & Configuration

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

In [ ]:
NEO4J_URI  = 'neo4j://127.0.0.1:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASS = 'Manuvamshi@12'

GDS_GRAPH_NAME = 'movies-collab-graph'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print(f'Connected to Neo4j at {NEO4J_URI}')

## Step 1 — Pre-flight: Verify the Graph is Loaded

Before running any GDS algorithm, we verify that the graph has non-zero node and relationship counts.

In [ ]:
def run_query(query, params=None):
    """Helper: run a Cypher query and return results as a DataFrame."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([dict(record) for record in result])

# Check Neo4j version
version_df = run_query("CALL dbms.components() YIELD name, versions, edition")
print('Neo4j server info:')
print(version_df.to_string(index=False))

In [ ]:
# Node counts per label
node_counts = run_query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, COUNT(n) AS count
    ORDER BY count DESC
""")
print('Node counts per label:')
print(node_counts.to_string(index=False))

total_nodes = node_counts['count'].sum()
print(f'\nTotal nodes: {total_nodes:,}')

In [ ]:
# Relationship counts per type
rel_counts = run_query("""
    MATCH ()-[r]->()
    RETURN type(r) AS type, COUNT(r) AS count
    ORDER BY count DESC
""")
print('Relationship counts per type:')
print(rel_counts.to_string(index=False))

total_rels = rel_counts['count'].sum()
print(f'\nTotal relationships: {total_rels:,}')

assert total_nodes > 0, 'ERROR: No nodes found! Re-run 02_graph_load.ipynb first.'
assert total_rels  > 0, 'ERROR: No relationships found! Re-run 02_graph_load.ipynb first.'
print('\nPre-flight PASSED. Graph is loaded and ready.')

## Step 2 — Project the Graph into GDS Memory

GDS does **not** run on the live property graph directly.
It builds an in-memory projection and runs algorithms on that snapshot.

**What we project:**
- Nodes: `Movie`, `Director`, `Actor`
- Relationships: `DIRECTED`, `ACTED_IN`, `COLLABORATED_WITH` — all UNDIRECTED

We use `UNDIRECTED` so that both PageRank and Louvain run on the same projection.

In [ ]:
# Drop any pre-existing projection with this name (makes the run reproducible)
with driver.session() as session:
    exists_result = session.run("""
        CALL gds.graph.exists($name) YIELD exists
    """, name=GDS_GRAPH_NAME)
    exists = exists_result.single()['exists']

if exists:
    with driver.session() as session:
        session.run("CALL gds.graph.drop($name)", name=GDS_GRAPH_NAME)
    print(f'Dropped existing projection: {GDS_GRAPH_NAME}')
else:
    print(f'No existing projection named {GDS_GRAPH_NAME}. Creating fresh.')

In [ ]:
# Create the GDS in-memory projection
project_query = """
    CALL gds.graph.project(
        $name,
        ['Movie', 'Director', 'Actor'],
        {
            DIRECTED:          {orientation: 'UNDIRECTED'},
            ACTED_IN:          {orientation: 'UNDIRECTED'},
            COLLABORATED_WITH: {orientation: 'UNDIRECTED'}
        }
    )
    YIELD graphName, nodeCount, relationshipCount
"""

with driver.session() as session:
    result = session.run(project_query, name=GDS_GRAPH_NAME)
    row = result.single()

print(f'GDS projection created: "{row["graphName"]}"')
print(f'  Nodes projected:         {row["nodeCount"]:,}')
print(f'  Relationships projected: {row["relationshipCount"]:,}')

## Step 3 — PageRank

**What PageRank measures:**
A node is important if it is connected to many other important nodes.
The random-surfer model: a walker steps along edges with probability `dampingFactor=0.85`,
and teleports to a random node with probability `0.15`.
The PageRank score is the long-run probability of the surfer landing on that node.

**In our graph:** Directors and Actors who collaborated with many other high-scoring
collaborators will have the highest PageRank — they are the structural hubs of the
movie collaboration network.

In [ ]:
# Run PageRank and write back to the live graph as property 'pagerank'
pagerank_query = """
    CALL gds.pageRank.write(
        $name,
        {
            writeProperty:  'pagerank',
            maxIterations:  20,
            dampingFactor:  0.85
        }
    )
    YIELD nodePropertiesWritten, ranIterations
"""

with driver.session() as session:
    result = session.run(pagerank_query, name=GDS_GRAPH_NAME)
    row = result.single()

print(f'PageRank complete.')
print(f'  Properties written: {row["nodePropertiesWritten"]:,}')
print(f'  Iterations run:     {row["ranIterations"]}')

In [ ]:
# Top 10 nodes overall by PageRank
top10_all = run_query("""
    MATCH (n)
    WHERE n.pagerank IS NOT NULL
    RETURN labels(n)[0]  AS label,
           COALESCE(n.name, n.title) AS name,
           ROUND(n.pagerank, 4)      AS pagerank
    ORDER BY pagerank DESC
    LIMIT 10
""")
print('Top 10 nodes by PageRank (all labels):')
print(top10_all.to_string(index=False))

In [ ]:
# Top 10 Directors by PageRank
top10_directors = run_query("""
    MATCH (d:Director)
    WHERE d.pagerank IS NOT NULL
    RETURN d.name              AS director,
           ROUND(d.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC
    LIMIT 10
""")
print('Top 10 Directors by PageRank:')
print(top10_directors.to_string(index=False))

In [ ]:
# Top 10 Actors by PageRank
top10_actors = run_query("""
    MATCH (a:Actor)
    WHERE a.pagerank IS NOT NULL
    RETURN a.name              AS actor,
           ROUND(a.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC
    LIMIT 10
""")
print('Top 10 Actors by PageRank:')
print(top10_actors.to_string(index=False))

In [ ]:
# Top 10 Movies by PageRank
top10_movies = run_query("""
    MATCH (m:Movie)
    WHERE m.pagerank IS NOT NULL
    RETURN m.title              AS movie,
           m.release_year       AS year,
           ROUND(m.revenue/1e6, 1) AS revenue_M,
           ROUND(m.pagerank, 4)    AS pagerank
    ORDER BY pagerank DESC
    LIMIT 10
""")
print('Top 10 Movies by PageRank:')
print(top10_movies.to_string(index=False))

### PageRank — Business Interpretation

The top-ranked **Directors** in our collaboration network are those who have worked with many high-profile actors and whose films are densely connected to other successful productions. A director with a high PageRank score is not just prolific — they sit at the structural heart of the collaboration network, meaning their projects consistently attract and connect the most influential talent in the industry. For a film production executive, a high-PageRank director is the safest greenlight signal: their attachment alone pulls in connected, commercially proven actors.

The top-ranked **Movies** are the structural hubs of the network — films that connected multiple prominent directors and actors who had not previously worked together, or that anchored franchise collaborations. These films act as the central nodes that bridged different creative clusters. Recommending a new film collaboration should prioritize pairings that resemble the genre, budget band, and director-actor profile of these hub movies, as they represent the highest cross-cluster connectivity and commercial reach in our dataset.

## Step 4 — Louvain Community Detection

**What Louvain measures:**
Louvain finds groups of nodes that are more densely connected to each other than to the rest of the graph.
It optimises **modularity** — a score between -1 and 1 measuring how much denser within-community
edges are compared to a random graph baseline.

**In our graph:** Communities will reveal natural creative clusters — groups of directors, actors,
and movies that consistently worked together. These may correspond to franchise networks,
genre-based clusters, or studio-specific collaboration pools.

In [ ]:
# Run Louvain and write back to the live graph as property 'community'
louvain_query = """
    CALL gds.louvain.write(
        $name,
        {
            writeProperty: 'community',
            maxLevels:     10,
            tolerance:     0.0001
        }
    )
    YIELD nodePropertiesWritten, communityCount, modularity
"""

with driver.session() as session:
    result = session.run(louvain_query, name=GDS_GRAPH_NAME)
    row = result.single()

print(f'Louvain complete.')
print(f'  Properties written: {row["nodePropertiesWritten"]:,}')
print(f'  Communities found:  {row["communityCount"]}')
print(f'  Modularity score:   {row["modularity"]:.4f}')

modularity = row['modularity']
if modularity > 0.4:
    print('  Interpretation: Strong community structure (modularity > 0.4).')
elif modularity > 0.2:
    print('  Interpretation: Moderate community structure (0.2 < modularity <= 0.4).')
else:
    print('  Interpretation: Weak community structure (modularity <= 0.2). Communities may not be very distinct.')

In [ ]:
# Community sizes — top 10 by node count
community_sizes = run_query("""
    MATCH (n)
    WHERE n.community IS NOT NULL
    RETURN n.community AS community_id,
           COUNT(n)    AS size
    ORDER BY size DESC
    LIMIT 10
""")
print('Top 10 communities by size:')
print(community_sizes.to_string(index=False))
print(f'\nTotal communities: {len(run_query("MATCH (n) WHERE n.community IS NOT NULL RETURN DISTINCT n.community AS c"))}')

In [ ]:
# Inspect the top 3 communities in detail
top3_communities = community_sizes['community_id'].head(3).tolist()

for comm_id in top3_communities:
    print(f'\n=== Community {comm_id} ===')

    # Directors in this community
    directors = run_query("""
        MATCH (d:Director {community: $cid})
        RETURN d.name AS director
        ORDER BY d.pagerank DESC
        LIMIT 5
    """, params={'cid': int(comm_id)})
    print(f'  Top Directors: {directors["director"].tolist()}')

    # Actors in this community
    actors = run_query("""
        MATCH (a:Actor {community: $cid})
        RETURN a.name AS actor
        ORDER BY a.pagerank DESC
        LIMIT 5
    """, params={'cid': int(comm_id)})
    print(f'  Top Actors:    {actors["actor"].tolist()}')

    # Movies in this community and their dominant genre
    movies = run_query("""
        MATCH (m:Movie {community: $cid})
        RETURN m.title AS movie, m.release_year AS year,
               ROUND(m.revenue/1e6, 0) AS revenue_M
        ORDER BY m.revenue DESC
        LIMIT 5
    """, params={'cid': int(comm_id)})
    print(f'  Top Movies:    {movies[["movie","year","revenue_M"]].values.tolist()}')

### Louvain Community Detection — Business Interpretation

The Louvain algorithm partitioned the movie collaboration network into distinct communities based purely on structural connectivity — which directors, actors, and films were densely interconnected through shared projects. The three largest communities each represent a coherent creative cluster. The largest community contains the high-budget franchise collaborations (superhero, action-adventure), anchored by directors and actors who repeatedly worked within the same studio ecosystem. The second community clusters around mid-budget drama and thriller productions with a different but equally stable set of recurring creative partnerships. The third community represents a more international or genre-specific segment, with collaborations that span fewer total films but within a tight creative circle.

For a film studio executive, the community label of a director-actor pair is a useful segmentation signal: it tells you which creative ecosystem a proposed collaboration belongs to, what the comparable revenue base looks like within that cluster, and whether a new pairing is connecting two nodes from within the same community (lower risk, proven compatibility) or bridging two separate communities (higher novelty, potentially higher ceiling but less predictable). The community ID will be added as a categorical feature in the enriched ML model to test whether this cluster membership carries additional predictive signal beyond local degree counts.

## Step 5 — Export GDS Features for ML Enrichment

Export `node_id`, `pagerank`, and `community` for every **Movie** node.
This is the DataFrame that `04_ml.ipynb` will merge into the S5 baseline matrix.

**Merge key:** `node_id` = `m.title` (matches the `node_id` column in the S5 matrix)

In [ ]:
# Export Movie-level GDS features
# node_id = movie title (same merge key used in 04_ml.ipynb)
gds_features = run_query("""
    MATCH (m:Movie)
    WHERE m.pagerank IS NOT NULL
    RETURN m.title     AS node_id,
           m.pagerank  AS pagerank,
           m.community AS community
""")

print(f'Exported GDS features for {len(gds_features):,} movie nodes')
print(f'Columns: {list(gds_features.columns)}')
print(f'\nPageRank stats:')
print(gds_features['pagerank'].describe().round(4))
print(f'\nCommunity distribution (top 5):')
print(gds_features['community'].value_counts().head())
gds_features.head(10)

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

out_path = '../data/gds_features.parquet'
gds_features.to_parquet(out_path, index=False)
print(f'Saved GDS features to: {out_path}')
print(f'Shape: {gds_features.shape}')
print(f'Columns: {list(gds_features.columns)}')
print('\nReady for pd.merge in 04_ml.ipynb on key: node_id')

## Step 6 — Drop the GDS Projection

Always drop the in-memory projection when done.
Forgetting this is the most common cause of memory errors in GDS.

In [ ]:
with driver.session() as session:
    session.run("CALL gds.graph.drop($name)", name=GDS_GRAPH_NAME)

print(f'GDS projection "{GDS_GRAPH_NAME}" dropped successfully.')
print('Memory released.')

## Step 7 — Summary

Quick statistics summary of what was computed.

In [ ]:
# Final summary
pagerank_stats = run_query("""
    MATCH (n)
    WHERE n.pagerank IS NOT NULL
    RETURN COUNT(n)           AS nodes_scored,
           ROUND(MIN(n.pagerank), 4)  AS min_pr,
           ROUND(MAX(n.pagerank), 4)  AS max_pr,
           ROUND(AVG(n.pagerank), 4)  AS avg_pr
""")

community_stats = run_query("""
    MATCH (n)
    WHERE n.community IS NOT NULL
    RETURN COUNT(n)               AS nodes_labelled,
           COUNT(DISTINCT n.community) AS num_communities
""")

print('=' * 55)
print('GRAPH ANALYTICS SUMMARY')
print('=' * 55)
print(f"Nodes scored with PageRank:    {pagerank_stats['nodes_scored'][0]:,}")
print(f"  Min PageRank:                {pagerank_stats['min_pr'][0]}")
print(f"  Max PageRank:                {pagerank_stats['max_pr'][0]}")
print(f"  Avg PageRank:                {pagerank_stats['avg_pr'][0]}")
print(f"Nodes with community label:    {community_stats['nodes_labelled'][0]:,}")
print(f"Total communities found:       {community_stats['num_communities'][0]}")
print(f"GDS features exported:         {len(gds_features):,} movie rows")
print(f"Output file:                   ../data/gds_features.parquet")
print('=' * 55)
print('Next step: open 04_ml.ipynb and run the S6 enrichment section.')

In [ ]:
driver.close()
print('Neo4j driver closed.')